In [2]:
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path=os.path.join(os.path.dirname(os.getcwd()), '.env'))
print('OpenAI key loaded:', bool(os.environ.get('OPENAI_API_KEY')))

OpenAI key loaded: True


## RAG 1. Install dependencies

In [3]:
!python -m pip install llama-index llama-index-llms-openai llama-index-embeddings-openai openai

zsh:1: command not found: python


## RAG 2. Configuration

In [4]:
import os

# Path to the NEURON repo docs folder (relative to doc-project/)
DOCS_PATHS = [
    "../docs/nmodl",   
    "../docs/progref",
]

# Where to save the persistent index (so you don't re-index every time)
INDEX_STORE_PATH = "neuron_index"

## Step 1. Summarize Threads

In [ ]:
import json
from llama_index.llms.openai import OpenAI

SUMMARIZE_PROMPT = """
You are helping to process NEURON simulator forum threads for documentation purposes.

Read the following forum thread and extract:
1. A brief summary of the problem and solution (2-3 sentences)
2. A list of NEURON functions, methods, or classes mentioned (e.g. CVode.event, NetCon.record, fadvance)

Return ONLY a JSON object with exactly these fields:
{
  "summary": "...",
  "functions_mentioned": ["...", "..."]
}
"""

with open("d_lst_posts.json", "r", encoding="utf-8") as f:
    threads = json.load(f)

llm = OpenAI(model="gpt-4o", temperature=0.0)
summaries = []

for item in threads:
    thread_id = item["id"]
    thread_text = item["post"]
    print(f"Summarizing thread {thread_id}...")
    try:
        prompt = SUMMARIZE_PROMPT + f"\n\n<thread>\n{thread_text}\n</thread>"
        raw = str(llm.complete(prompt)).strip()
        if raw.startswith("```"):
            raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
        result = json.loads(raw)
        summaries.append({
            "thread_id": thread_id,
            "summary": result.get("summary", ""),
            "functions_mentioned": result.get("functions_mentioned", [])
        })
    except Exception as e:
        print(f"ERROR: thread {thread_id}: {e}")
        summaries.append({
            "thread_id": thread_id,
            "summary": "",
            "functions_mentioned": []
        })

print(f"\nTotal summaries: {len(summaries)}")

with open("summaries.json", "w", encoding="utf-8") as f:
    json.dump(summaries, f, indent=2)
print("Saved to summaries.json")

## Step 2. Chunk RST Sections

In [ ]:
import os
import json
import re

CHUNK_DOCS_PATHS = [
    "../docs/nmodl",
    "../docs/progref",
]

# RST heading underline characters in conventional hierarchy order
HEADING_HIERARCHY = ['#', '*', '=', '-', '^', '"', '~', '+']

def get_heading_level(underline_char):
    """Map RST underline character to an integer heading level."""
    if underline_char in HEADING_HIERARCHY:
        return HEADING_HIERARCHY.index(underline_char) + 1
    return len(HEADING_HIERARCHY) + 1

def is_rst_heading(lines, line_idx):
    """Return True if lines[line_idx] is an RST section heading."""
    if line_idx < 0 or line_idx + 1 >= len(lines):
        return False
    line = lines[line_idx]
    next_line = lines[line_idx + 1]
    return (
        len(line.strip()) > 0 and
        len(next_line.strip()) >= len(line.strip()) and
        bool(re.match(r'^[=\-~^#*+`\'"_]{3,}$', next_line.strip()))
    )

def chunk_rst_file(file_path):
    """Split an RST file into sections by heading."""
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.read().split('\n')

    # Find all heading positions
    heading_positions = []  # (line_idx, heading_text, underline_char)
    for i in range(len(lines) - 1):
        if is_rst_heading(lines, i):
            underline_char = lines[i + 1][0] if lines[i + 1].strip() else '='
            heading_positions.append((i, lines[i].strip(), underline_char))

    if not heading_positions:
        # No headings — treat whole file as one chunk
        return [{
            "file_path": file_path,
            "section_heading": os.path.basename(file_path),
            "heading_level": 1,
            "content": "\n".join(lines).strip()
        }]

    chunks = []
    for idx, (line_idx, heading_text, underline_char) in enumerate(heading_positions):
        # Content starts after the heading line and its underline
        section_start = line_idx + 2
        # Content ends at the next heading or end of file
        section_end = heading_positions[idx + 1][0] if idx + 1 < len(heading_positions) else len(lines)
        content = "\n".join(lines[section_start:section_end]).strip()
        chunks.append({
            "file_path": file_path,
            "section_heading": heading_text,
            "heading_level": get_heading_level(underline_char),
            "content": content
        })

    return chunks


# Walk docs directories and chunk all RST files
all_chunks = []
for docs_path in CHUNK_DOCS_PATHS:
    if not os.path.exists(docs_path):
        print(f"WARNING: {docs_path} does not exist, skipping.")
        continue
    for root, dirs, files in os.walk(docs_path):
        for fname in sorted(files):
            if fname.endswith('.rst'):
                file_path = os.path.join(root, fname)
                try:
                    chunks = chunk_rst_file(file_path)
                    all_chunks.extend(chunks)
                except Exception as e:
                    print(f"ERROR: {file_path}: {e}")

print(f"Total chunks: {len(all_chunks)}")

# Save to chunks.json
with open("chunks.json", "w", encoding="utf-8") as f:
    json.dump(all_chunks, f, indent=2)
print("Saved to chunks.json")

## RAG 3. Build (or load) the index

In [5]:
from llama_index.core import (
    VectorStoreIndex,
    SimpleDirectoryReader,
    StorageContext,
    load_index_from_storage,
)
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core import Settings

# Configure models
Settings.llm = OpenAI(model="gpt-4o", temperature=0.2)
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

if os.path.exists(INDEX_STORE_PATH):
    print("Loading existing index...")
    storage_context = StorageContext.from_defaults(persist_dir=INDEX_STORE_PATH)
    index = load_index_from_storage(storage_context)
else:
    print("Building index from docs (this may take a few minutes)...")
    all_documents = []
    for docs_path in DOCS_PATHS:
        docs = SimpleDirectoryReader(
            input_dir=docs_path,
            recursive=True,
            required_exts=[".rst"],
        ).load_data()
        print(f"Loaded {len(docs)} .rst files from {docs_path}")
        all_documents.extend(docs)
    print(f"Total: {len(all_documents)} .rst files")
    index = VectorStoreIndex.from_documents(all_documents)
    index.storage_context.persist(persist_dir=INDEX_STORE_PATH)
    print("Index built and saved.")

Loading existing index...


## RAG 4. Define the prompt template

In [6]:
PROMPT_TEMPLATE = """
You are a technical documentation assistant helping integrate community
Q&A content into the NEURON simulator's official documentation.

Below are the most relevant excerpts from the current NEURON documentation,
each labelled with its source .rst file path:

---------------------
{context_str}
---------------------

Here is a forum Q&A thread that contains information to be integrated:

<thread>
{query_str}
</thread>

Instructions:
- Identify every function, method, or class that the thread adds new information about.
- For each one, produce a unified diff showing what should be added to the relevant .rst file.
- Use standard unified diff format:
    --- a/<rst_file_path>
    +++ b/<rst_file_path>
    @@ -<line>,<count> +<line>,<count> @@
     (context lines with a leading space)
    +(new lines with a leading +)
- Base diffs on the existing documentation excerpts shown above.
- New content must match RST style (directives, inline code, section structure).
- Only include a code block if you can provide a complete, working example — either from the thread or generated yourself. Never include a partial or placeholder code block.
- Use ``.. code-block:: python`` only for actual Python code. Use ``.. code-block:: none`` for HOC code or any other language.
- Where relevant, provide both a HOC and a Python example to support users of both interfaces.
- If multiple files need changes, concatenate their diffs.
- Never insert content in the middle of a function definition, parameter description, or syntax block. Only add content at the end of the most relevant section, or after the last related paragraph.

Return ONLY the unified diff text, with no preamble, explanation, or code fences.
Only include information clearly supported by the thread. Do not extrapolate.
"""

## RAG 5. Run a query for a single forum thread

In [7]:
import json
# Paste your forum thread here
with open("d_lst_posts.json", "r", encoding="utf-8") as f:
    entry = json.load(f)

In [ ]:
from llama_index.llms.openai import OpenAI
import re

def is_rst_heading(lines, line_idx):
    """Return True if lines[line_idx] is an RST section heading."""
    if line_idx < 0 or line_idx + 1 >= len(lines):
        return False
    line = lines[line_idx]
    next_line = lines[line_idx + 1]
    return (
        len(line.strip()) > 0 and
        len(next_line.strip()) >= len(line.strip()) and
        bool(re.match(r'^[=\-~^#*+`\'"_]{3,}$', next_line.strip()))
    )

def extract_section(file_path, chunk_text, max_lines=150):
    """
    Read file_path and return the RST section containing chunk_text.
    Falls back to chunk_text if the file cannot be read or chunk is not found.
    """
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            full_text = f.read()
    except OSError:
        return chunk_text

    lines = full_text.split('\n')
    search_text = chunk_text[:120].strip()
    chunk_pos = full_text.find(search_text)
    if chunk_pos == -1:
        return chunk_text

    chunk_line = full_text[:chunk_pos].count('\n')

    # Walk upward to find the nearest heading
    section_start = max(0, chunk_line - max_lines)
    for i in range(chunk_line, -1, -1):
        if is_rst_heading(lines, i):
            section_start = i
            break

    # Walk downward to find the next heading
    section_end = min(len(lines), chunk_line + max_lines)
    for i in range(chunk_line + 1, len(lines)):
        if is_rst_heading(lines, i):
            section_end = i
            break

    # Cap to max_lines to avoid sending huge sections
    if section_end - section_start > max_lines:
        section_end = section_start + max_lines

    return '\n'.join(lines[section_start:section_end])


def process_thread(thread_id, thread_text):
    """Query the index with a forum thread and return a unified diff string."""
    estimated_tokens = len(thread_text) // 4
    if estimated_tokens > 8000:
        print(f"WARNING: Thread {thread_id} is large (~{estimated_tokens} tokens) and may hit the token limit.")
    try:
        # Step 1: retrieve the most relevant chunks
        retriever = index.as_retriever(similarity_top_k=5)
        nodes = retriever.retrieve(thread_text)

        # Step 2: expand each chunk to its full RST section
        context_parts = []
        for node in nodes:
            file_path = node.metadata.get('file_path', '')
            chunk_text = node.get_content()
            section = extract_section(file_path, chunk_text)
            context_parts.append(f"Source: {file_path}\n\n{section}")

        context_str = "\n\n---\n\n".join(context_parts)

        # Step 3: fill the prompt and call GPT directly
        prompt = PROMPT_TEMPLATE.format(context_str=context_str, query_str=thread_text)
        llm = OpenAI(model="gpt-4o", temperature=0.2)
        raw = str(llm.complete(prompt)).strip()

        # Strip markdown code fences if present
        if raw.startswith("```"):
            raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()

        return raw

    except Exception as e:
        error_msg = str(e)
        if "maximum context length" in error_msg or "token" in error_msg.lower():
            print(f"ERROR: Thread {thread_id} exceeded token limit (~{estimated_tokens} tokens).")
        else:
            print(f"ERROR: Thread {thread_id} failed: {error_msg}")
        return None


# Test on the first thread
forum_id = entry[1]["id"]
forum_thread = entry[1]["post"]

result = process_thread(forum_id, forum_thread)
print(result)

## RAG 6. Bulk processing

In [ ]:
import os

OUTPUT_DIR = "doc-project/diffs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

for item in entry:
    thread_id = item['id']
    thread_text = item['post']
    print(f"Processing thread {thread_id}...")
    diff = process_thread(thread_id, thread_text)
    if diff:
        out_path = os.path.join(OUTPUT_DIR, f"thread_{thread_id}.diff")
        with open(out_path, "w", encoding="utf-8") as f:
            f.write(diff)
        print(f"  Saved: {out_path}")
    else:
        print(f"  Skipped thread {thread_id} (error or empty response).")

## RAG 7. Save results to a file for review

In [ ]:
# Diff files are saved to doc-project/diffs/thread_<id>.diff during bulk processing above.
# To apply a diff: patch -p1 < doc-project/diffs/thread_<id>.diff